In [14]:
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

file1 = 'ulasanRedbus_dengan_sentimen_labeling_1.csv'
file2 = 'ulasanRedbus_dengan_sentimen_labeling_2.csv'
file3 = 'ulasanRedbus_dengan_sentimen_labeling_3.csv'

df1 = pd.read_csv(file1).sort_values('reviewId').reset_index(drop=True)
df2 = pd.read_csv(file2).sort_values('reviewId').reset_index(drop=True)
df3 = pd.read_csv(file3).sort_values('reviewId').reset_index(drop=True)

aspek_penelitian = ['Ticketing', 'Information Channels', 'Travel Experience']

for aspect in aspek_penelitian:
    print(f"\n=========================================")
    print(f"Menganalisis Aspek: {aspect.upper()}")
    print(f"=========================================")
    
    r1 = df1[aspect].fillna('none').astype(str).str.lower().str.strip()
    r2 = df2[aspect].fillna('none').astype(str).str.lower().str.strip()
    r3 = df3[aspect].fillna('none').astype(str).str.lower().str.strip()
    
    df_cek = pd.DataFrame({
        'Review_ID': df1['reviewId'],
        'Annotator1': r1, 
        'Annotator2': r2, 
        'Annotator3': r3
    })
    
    kondisi_bukan_aspek = (df_cek['Annotator1'] == 'none') & (df_cek['Annotator2'] == 'none') & (df_cek['Annotator3'] == 'none')
    df_aspek_terkait = df_cek[~kondisi_bukan_aspek].copy()
    

    total_ulasan_aspek = len(df_aspek_terkait)
    
    if total_ulasan_aspek == 0:
        print("PERINGATAN: Tidak ada ulasan yang membahas aspek ini sama sekali di ketiga file.")
        continue
        

    beda = df_aspek_terkait[
        (df_aspek_terkait['Annotator1'] != df_aspek_terkait['Annotator2']) | 
        (df_aspek_terkait['Annotator1'] != df_aspek_terkait['Annotator3']) | 
        (df_aspek_terkait['Annotator2'] != df_aspek_terkait['Annotator3'])
    ]
    
    
    if len(beda) > 0:
        print("\nContoh 5 baris pertama yang mengalami perbedaan penilaian:")
        print(beda.head().to_string(index=False))
        
        formatted_data, _ = aggregate_raters(df_aspek_terkait[['Annotator1', 'Annotator2', 'Annotator3']])
        kappa = fleiss_kappa(formatted_data)
        
        print(f"\n>>> SKOR FLEISS' KAPPA (Khusus Aspek '{aspect}'): {kappa:.4f}")
        print(f"Jumlah baris yang TIDAK SEPAKAT: {len(beda)} dari {total_ulasan_aspek} total ulasan valid pada aspek '{aspect}'.")
    else:
        print("\nINFORMASI: Ketiga rater 100% IDENTIK dan SEPAKAT pada seluruh ulasan di aspek ini!")
        print(f">>> SKOR FLEISS' KAPPA (Khusus Aspek '{aspect}'): 1.0000")


Menganalisis Aspek: TICKETING

Contoh 5 baris pertama yang mengalami perbedaan penilaian:
                           Review_ID Annotator1 Annotator2 Annotator3
00ce5abb-0dad-4b07-abc7-d14730f16f30       none    negatif    negatif
00f57ca9-624a-4c91-b13c-a7bd9e7dba13    positif       none    positif
00ff6c8b-e0e1-4440-a286-8c91c5f481a6    negatif       none    negatif
029ba514-a17e-4cf3-91f9-176bec7ab677    negatif       none       none
02ac195c-3170-41da-ac84-487397707509       none    positif       none

>>> SKOR FLEISS' KAPPA (Khusus Aspek 'Ticketing'): 0.6880
Jumlah baris yang TIDAK SEPAKAT: 246 dari 884 total ulasan valid pada aspek 'Ticketing'.

Menganalisis Aspek: INFORMATION CHANNELS

Contoh 5 baris pertama yang mengalami perbedaan penilaian:
                           Review_ID Annotator1 Annotator2 Annotator3
003ffefa-3944-4e30-96ae-0642433ba2f3       none    negatif       none
00916aaa-4373-41be-b786-e6b174f50f9a       none    negatif    negatif
00b0c414-4d18-439d-a694-e4453

In [15]:
import pandas as pd
from scipy.stats import mode

df1 = pd.read_csv('ulasanRedbus_dengan_sentimen_labeling_1.csv').sort_values('reviewId').reset_index(drop=True)
df2 = pd.read_csv('ulasanRedbus_dengan_sentimen_labeling_2.csv').sort_values('reviewId').reset_index(drop=True)
df3 = pd.read_csv('ulasanRedbus_dengan_sentimen_labeling_3.csv').sort_values('reviewId').reset_index(drop=True)

df_final = pd.DataFrame()
df_final['reviewId'] = df1['reviewId']
df_final['review'] = df1['review']

aspects = ['Ticketing', 'Information Channels', 'Travel Experience']

for aspect in aspects:
    r1 = df1[aspect].fillna('none').str.lower().str.strip()
    r2 = df2[aspect].fillna('none').str.lower().str.strip()
    r3 = df3[aspect].fillna('none').str.lower().str.strip()
    
    combined = pd.DataFrame({'R1': r1, 'R2': r2, 'R3': r3})
    
    majority_vote = combined.mode(axis=1)[0]
    
    df_final[aspect] = majority_vote.replace('none', None)

df_final.to_csv('Redbus_Ground_Truth_Final.csv', index=False)